In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e8/train.csv
/kaggle/input/competitions/playground-series-s6e8/test.csv


In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/test.csv')
sample_submission = pd.read_csv(
    '/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv')
                    

In [4]:
print("Train:", train.shape)
print("Test:", test.shape)
print("Submission:", sample_submission.shape)

Train: (691369, 14)
Test: (296302, 13)
Submission: (296302, 2)


In [ ]:
display(train.head())
display(test.head())
display(sample_submission.head())

In [ ]:
train.info()

In [ ]:
train.describe(include='all').T

In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

In [ ]:
print("Train dtypes:")
display(train.dtypes)

In [ ]:
train_only = set(train.columns) - set(test.columns)

print("Columns only in train:")
print(train_only)

In [ ]:
missing = train.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

display(missing)

In [ ]:
missing_test = test.isnull().sum()

missing_test = missing_test[missing_test > 0].sort_values(ascending=False)

display(missing_test)

In [ ]:
print("Duplicate rows in train:", train.duplicated().sum())
print("Duplicate rows in test:", test.duplicated().sum())

In [ ]:
print("Unique train IDs:", train['id'].nunique())
print("Train rows:", len(train))

print("Unique test IDs:", test['id'].nunique())
print("Test rows:", len(test))

In [ ]:
categorical_cols = train.select_dtypes(
    include=['object', 'category']
).columns

print("Categorical columns:")
print(categorical_cols.tolist())

In [ ]:
for col in categorical_cols:
    print(f"\n===== {col} =====")
    print(train[col].value_counts(dropna=False).head(20))

In [ ]:
numerical_cols = train.select_dtypes(
    include='number'
).columns

print("Numerical columns:")
print(numerical_cols.tolist())

In [ ]:
display(train[numerical_cols].describe().T)

In [ ]:
numerical_cols = test.select_dtypes(
    include='number'
).columns

print("Numerical columns:")
print(numerical_cols.tolist())

In [ ]:
display(test[numerical_cols].describe().T)

In [ ]:
# Count and percentage of each target value
target_summary = pd.DataFrame({
    "Count": train["addicted_label"].value_counts(),
    "Percentage": train["addicted_label"].value_counts(normalize=True) * 100
})

display(target_summary)

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": train.isnull().sum(),
    "Missing Percentage": train.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "Missing Percentage",
    ascending=False
)

display(missing_summary)

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": test.isnull().sum(),
    "Missing Percentage": test.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "Missing Percentage",
    ascending=False
)

display(missing_summary)

In [ ]:
target = "addicted_label"

missing_target_analysis = []

for col in train.columns:
    if col == target:
        continue
    
    missing_mask = train[col].isna()
    
    if missing_mask.sum() > 0:
        missing_target_analysis.append({
            "Feature": col,
            "Missing_Count": missing_mask.sum(),
            "Missing_%": missing_mask.mean() * 100,
            "Addiction_%_When_Missing": train.loc[missing_mask, target].mean() * 100,
            "Addiction_%_When_Not_Missing": train.loc[~missing_mask, target].mean() * 100
        })

missing_target_analysis = pd.DataFrame(missing_target_analysis)

display(
    missing_target_analysis.sort_values(
        "Addiction_%_When_Missing",
        ascending=False
    )
)

In [ ]:
numerical_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

target_stats = train.groupby("addicted_label")[numerical_features].mean().T

display(target_stats)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numerical_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

# Sample data for faster plotting
plot_data = train.sample(
    n=min(100000, len(train)),
    random_state=42
)

plt.figure(figsize=(18, 12))

for i, col in enumerate(numerical_features, 1):
    plt.subplot(3, 3, i)
    
    sns.boxplot(
        data=plot_data,
        x="addicted_label",
        y=col
    )
    
    plt.title(col)
    plt.xlabel("Addicted Label")

plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = train[numerical_features + ["addicted_label"]].corr()

display(correlation_matrix.round(3))

In [ ]:
plt.figure(figsize=(12, 9))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
categorical_features = [
    "gender",
    "stress_level",
    "academic_work_impact"
]

for col in categorical_features:
    print(f"\n{'='*60}")
    print(f"{col}")
    print(f"{'='*60}")
    
    summary = (
        train.groupby(col, dropna=False)["addicted_label"]
        .agg(["count", "mean"])
    )
    
    summary["addiction_percentage"] = summary["mean"] * 100
    
    display(summary)

In [ ]:
pd.crosstab(
    pd.qcut(train["daily_screen_time_hours"], q=10, duplicates="drop"),
    train["addicted_label"],
    normalize="index"
).round(3)

In [ ]:
pd.crosstab(
    pd.qcut(
        train["weekend_screen_time"],
        q=10,
        duplicates="drop"
    ),
    train["addicted_label"],
    normalize="index"
).round(3)

In [ ]:
pd.crosstab(
    pd.qcut(
        train["social_media_hours"],
        q=10,
        duplicates="drop"
    ),
    train["addicted_label"],
    normalize="index"
).round(3)

In [ ]:
temp = train[
    ["daily_screen_time_hours",
     "social_media_hours",
     "addicted_label"]
].dropna()

temp["screen_bin"] = pd.qcut(
    temp["daily_screen_time_hours"],
    q=5,
    duplicates="drop"
)

temp["social_bin"] = pd.qcut(
    temp["social_media_hours"],
    q=5,
    duplicates="drop"
)

pivot = pd.pivot_table(
    temp,
    values="addicted_label",
    index="screen_bin",
    columns="social_bin",
    aggfunc="mean",
    observed=False
)

display(pivot.round(3))

In [ ]:
print("Duplicate complete rows in train:", train.duplicated().sum())

print(
    "Duplicate feature rows excluding id and target:",
    train.drop(columns=["id", "addicted_label"]).duplicated().sum()
)

print("Duplicate IDs in train:", train["id"].duplicated().sum())
print("Duplicate IDs in test:", test["id"].duplicated().sum())

In [ ]:
for col in numerical_features:
    print(f"\n{'='*60}")
    print(col)
    print(f"{'='*60}")
    
    print(train[col].quantile([
        0.001,
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999
    ]))

In [ ]:
missing_corr = train.isna().corr()

plt.figure(figsize=(12, 9))

sns.heatmap(
    missing_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between Missingness Patterns")
plt.tight_layout()
plt.show()

In [ ]:
import sklearn
print("scikit-learn:", sklearn.__version__)

try:
    import xgboost
    print("xgboost:", xgboost.__version__)
except:
    print("xgboost not available")

try:
    import lightgbm
    print("lightgbm:", lightgbm.__version__)
except:
    print("lightgbm not available")

try:
    import catboost
    print("catboost:", catboost.__version__)
except:
    print("catboost not available")

In [ ]:
non_negative_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

negative_counts = train[non_negative_features].lt(0).sum()

print("Negative values in TRAIN:")
display(negative_counts)

In [ ]:
print("Negative values in TEST:")
display(test[non_negative_features].lt(0).sum())

In [ ]:
expected_ranges = {
    "age": (18, 35),
    "daily_screen_time_hours": (0.5, 15),
    "social_media_hours": (0, 8),
    "gaming_hours": (0, 4),
    "work_study_hours": (0, 6),
    "sleep_hours": (4.5, 9),
    "notifications_per_day": (20, 250),
    "app_opens_per_day": (15, 180),
    "weekend_screen_time": (0.5, 17.56)
}

for col, (lower, upper) in expected_ranges.items():
    train_invalid = (
        (train[col] < lower) |
        (train[col] > upper)
    ).sum()

    test_invalid = (
        (test[col] < lower) |
        (test[col] > upper)
    ).sum()

    print(
        f"{col:30s} "
        f"Train invalid: {train_invalid:6d} | "
        f"Test invalid: {test_invalid:6d}"
    )

In [ ]:
categorical_features = [
    "gender",
    "stress_level",
    "academic_work_impact"
]

for col in categorical_features:
    print(f"\n{'='*50}")
    print(col)
    print("TRAIN:")
    print(train[col].value_counts(dropna=False))

    print("\nTEST:")
    print(test[col].value_counts(dropna=False))

In [ ]:
import numpy as np

print("Infinite values in TRAIN:")
display(
    np.isinf(train.select_dtypes(include=np.number)).sum()
)

print("Infinite values in TEST:")
display(
    np.isinf(test.select_dtypes(include=np.number)).sum()
)

In [ ]:
print(train["addicted_label"].value_counts(dropna=False))
print("Unique target values:", train["addicted_label"].unique())
print("Target missing:", train["addicted_label"].isna().sum())

In [ ]:
train["activity_hours_sum"] = (
    train["social_media_hours"]
    + train["gaming_hours"]
    + train["work_study_hours"]
)

display(
    train[
        [
            "daily_screen_time_hours",
            "social_media_hours",
            "gaming_hours",
            "work_study_hours",
            "activity_hours_sum"
        ]
    ].describe()
)

In [ ]:
activity_sum = (
    train["social_media_hours"]
    + train["gaming_hours"]
    + train["work_study_hours"]
)

comparison = pd.DataFrame({
    "daily_screen_time": train["daily_screen_time_hours"],
    "activity_sum": activity_sum
})

# Only rows where all four values are available
comparison_clean = comparison.dropna()

comparison_clean["difference"] = (
    comparison_clean["daily_screen_time"]
    - comparison_clean["activity_sum"]
)

display(comparison_clean["difference"].describe())

In [ ]:
print(
    "Rows where social + gaming + work/study > daily screen time:",
    (comparison_clean["difference"] < 0).sum()
)

print(
    "Percentage:",
    (comparison_clean["difference"] < 0).mean() * 100
)

In [ ]:
# Divide training IDs into 20 equal-sized ranges
train_id_analysis = train.copy()

train_id_analysis["id_bin"] = pd.qcut(
    train_id_analysis["id"],
    q=20,
    duplicates="drop"
)

id_target_analysis = (
    train_id_analysis
    .groupby("id_bin", observed=True)["addicted_label"]
    .agg(["count", "mean"])
)

id_target_analysis["mean"] *= 100

display(id_target_analysis)

In [ ]:
print(
    "Correlation between ID and target:",
    train["id"].corr(train["addicted_label"])
)

In [ ]:
important_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

id_feature_corr = train[
    ["id"] + important_features
].corr()["id"].drop("id").sort_values()

display(id_feature_corr)

In [ ]:
import numpy as np
import pandas as pd

# Keep test IDs ONLY for the final Kaggle submission
test_ids = test["id"].copy()

# Drop ID from model features
X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

X_test = test.drop(columns=["id"])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

In [ ]:
numerical_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

categorical_features = [
    "gender",
    "stress_level",
    "academic_work_impact"
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    )
])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )
    )
])

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

print("\nTarget distribution:")
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)

In [4]:
# ============================================================
# CATBOOST DATA PREPARATION
# ============================================================

X_cb = train.drop(columns=["id", "addicted_label"]).copy()
y_cb = train["addicted_label"].copy()

X_test_cb = test.drop(columns=["id"]).copy()

test_ids = test["id"].copy()

categorical_features = [
    "gender",
    "stress_level",
    "academic_work_impact"
]

# CatBoost requires categorical values to be strings.
# Replace categorical NaNs with an explicit category.
for col in categorical_features:
    X_cb[col] = X_cb[col].fillna("Missing").astype(str)
    X_test_cb[col] = X_test_cb[col].fillna("Missing").astype(str)

print("Training shape:", X_cb.shape)
print("Test shape:", X_test_cb.shape)

print("\nCategorical missing values:")
print(X_cb[categorical_features].isna().sum())

Training shape: (691369, 12)
Test shape: (296302, 12)

Categorical missing values:
gender                  0
stress_level            0
academic_work_impact    0
dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X_train_cb, X_val_cb, y_train_cb, y_val_cb = train_test_split(
    X_cb,
    y_cb,
    test_size=0.20,
    stratify=y_cb,
    random_state=42
)

In [ ]:
from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False
)

catboost_model.fit(
    X_train_cb,
    y_train_cb,
    cat_features=categorical_features,
    eval_set=(X_val_cb, y_val_cb),
    early_stopping_rounds=50
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

y_val_cb_prob = catboost_model.predict_proba(X_val_cb)[:, 1]
y_val_cb_pred = (y_val_cb_prob >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_val_cb, y_val_cb_pred))
print("ROC-AUC:", roc_auc_score(y_val_cb, y_val_cb_prob))

print("\nClassification Report:")
print(classification_report(y_val_cb, y_val_cb_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_cb, y_val_cb_pred))

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np
import time

# ------------------------------------------------------------
# 5-Fold Stratified Cross-Validation
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_cb, y_cb), 1
):

    print("\n" + "=" * 60)
    print(f"FOLD {fold}/5")
    print("=" * 60)

    X_fold_train = X_cb.iloc[train_idx]
    X_fold_val = X_cb.iloc[val_idx]

    y_fold_train = y_cb.iloc[train_idx]
    y_fold_val = y_cb.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    start_time = time.time()

    model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=categorical_features,
        eval_set=(X_fold_val, y_fold_val),
        early_stopping_rounds=50
    )

    val_prob = model.predict_proba(X_fold_val)[:, 1]

    fold_auc = roc_auc_score(
        y_fold_val,
        val_prob
    )

    fold_scores.append(fold_auc)

    elapsed = time.time() - start_time

    print(f"\nFold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} time: {elapsed / 60:.2f} minutes")


print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)

for i, score in enumerate(fold_scores, 1):
    print(f"Fold {i}: {score:.6f}")

print(f"\nMean ROC-AUC: {np.mean(fold_scores):.6f}")
print(f"Std ROC-AUC:  {np.std(fold_scores):.6f}")

In [ ]:
best_fold = np.argmax(fold_scores) + 1
best_auc = max(fold_scores)

print(f"Best Fold: {best_fold}")
print(f"Best ROC-AUC: {best_auc:.6f}")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from catboost import CatBoostClassifier

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Get the indices for the best fold
for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_cb, y_cb), 1
):
    if fold == best_fold:
        best_train_idx = train_idx
        best_val_idx = val_idx
        break

# Get best-fold data
X_best_train = X_cb.iloc[best_train_idx]
X_best_val = X_cb.iloc[best_val_idx]

y_best_train = y_cb.iloc[best_train_idx]
y_best_val = y_cb.iloc[best_val_idx]

# Train the model with the same parameters
best_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False
)

best_model.fit(
    X_best_train,
    y_best_train,
    cat_features=categorical_features,
    eval_set=(X_best_val, y_best_val),
    early_stopping_rounds=50
)

# Predictions
best_prob = best_model.predict_proba(X_best_val)[:, 1]
best_pred = (best_prob >= 0.5).astype(int)

# Metrics
accuracy = accuracy_score(y_best_val, best_pred)
precision = precision_score(y_best_val, best_pred)
recall = recall_score(y_best_val, best_pred)
f1 = f1_score(y_best_val, best_pred)
roc_auc = roc_auc_score(y_best_val, best_prob)

print("\n" + "=" * 50)
print(f"BEST FOLD: {best_fold}")
print("=" * 50)

print(f"Accuracy : {accuracy:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall   : {recall:.6f}")
print(f"F1 Score : {f1:.6f}")
print(f"ROC-AUC  : {roc_auc:.6f}")

print("\nClassification Report:")
print(classification_report(y_best_val, best_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_best_val, best_pred))

In [ ]:
catboost_long = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False
)

catboost_long.fit(
    X_train_cb,
    y_train_cb,
    cat_features=categorical_features,
    eval_set=(X_val_cb, y_val_cb),
    early_stopping_rounds=100
)

In [ ]:
y_prob_long = catboost_long.predict_proba(X_val_cb)[:, 1]

print("ROC-AUC:", roc_auc_score(y_val_cb, y_prob_long))
print("Best iteration:", catboost_long.get_best_iteration())
print("Best validation AUC:", catboost_long.get_best_score())

In [ ]:
catboost_1500 = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False
)

catboost_1500.fit(
    X_train_cb,
    y_train_cb,
    cat_features=categorical_features,
    eval_set=(X_val_cb, y_val_cb),
    early_stopping_rounds=150
)

y_prob_1500 = catboost_1500.predict_proba(X_val_cb)[:, 1]

print("ROC-AUC:", roc_auc_score(y_val_cb, y_prob_1500))
print("Best iteration:", catboost_1500.get_best_iteration())
print("Best validation score:")
print(catboost_1500.get_best_score())

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np
import time

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores_1500 = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_cb, y_cb), 1
):

    print("\n" + "=" * 60)
    print(f"FOLD {fold}/5")
    print("=" * 60)

    X_fold_train = X_cb.iloc[train_idx]
    X_fold_val = X_cb.iloc[val_idx]

    y_fold_train = y_cb.iloc[train_idx]
    y_fold_val = y_cb.iloc[val_idx]

    model_1500 = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=300,
        thread_count=-1,
        allow_writing_files=False
    )

    start_time = time.time()

    model_1500.fit(
        X_fold_train,
        y_fold_train,
        cat_features=categorical_features
    )

    val_prob = model_1500.predict_proba(X_fold_val)[:, 1]

    fold_auc = roc_auc_score(
        y_fold_val,
        val_prob
    )

    fold_scores_1500.append(fold_auc)

    elapsed = time.time() - start_time

    print(f"\nFold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} time: {elapsed / 60:.2f} minutes")


print("\n" + "=" * 60)
print("1500 ITERATION CV RESULTS")
print("=" * 60)

for i, score in enumerate(fold_scores_1500, 1):
    print(f"Fold {i}: {score:.6f}")

print(f"\nMean ROC-AUC: {np.mean(fold_scores_1500):.6f}")
print(f"Std ROC-AUC:  {np.std(fold_scores_1500):.6f}")

In [7]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import time

depth_results = {}

for depth in [10, 11, 12, 13, 14]:

    print("\n" + "=" * 60)
    print(f"TESTING DEPTH = {depth}")
    print("=" * 60)

    model_depth = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=depth,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=500,
        thread_count=-1,
        allow_writing_files=False
    )

    start = time.time()

    model_depth.fit(
        X_train_cb,
        y_train_cb,
        cat_features=categorical_features,
        eval_set=(X_val_cb, y_val_cb),
        early_stopping_rounds=150
    )

    val_prob = model_depth.predict_proba(X_val_cb)[:, 1]

    auc = roc_auc_score(
        y_val_cb,
        val_prob
    )

    best_iteration = model_depth.get_best_iteration()

    elapsed = (time.time() - start) / 60

    depth_results[depth] = {
        "AUC": auc,
        "Best Iteration": best_iteration,
        "Time": elapsed
    }

    print(f"\nDepth {depth}")
    print(f"ROC-AUC: {auc:.6f}")
    print(f"Best iteration: {best_iteration}")
    print(f"Time: {elapsed:.2f} minutes")


print("\n" + "=" * 60)
print("DEPTH EXPERIMENT RESULTS")
print("=" * 60)

for depth, result in depth_results.items():
    print(
        f"Depth {depth}: "
        f"AUC={result['AUC']:.6f}, "
        f"Best Iteration={result['Best Iteration']}"
    )


TESTING DEPTH = 10
0:	test: 0.9160770	best: 0.9160770 (0)	total: 733ms	remaining: 18m 18s
500:	test: 0.9565495	best: 0.9565495 (500)	total: 4m 50s	remaining: 9m 38s
1000:	test: 0.9605804	best: 0.9605804 (1000)	total: 9m 51s	remaining: 4m 54s
1499:	test: 0.9618250	best: 0.9618269 (1498)	total: 14m 47s	remaining: 0us

bestTest = 0.9618269257
bestIteration = 1498

Shrink model to first 1499 iterations.

Depth 10
ROC-AUC: 0.961827
Best iteration: 1498
Time: 14.83 minutes

TESTING DEPTH = 11
0:	test: 0.9191388	best: 0.9191388 (0)	total: 726ms	remaining: 18m 7s
500:	test: 0.9571469	best: 0.9571469 (500)	total: 5m 51s	remaining: 11m 40s
1000:	test: 0.9607429	best: 0.9607429 (1000)	total: 11m 45s	remaining: 5m 51s
1499:	test: 0.9617391	best: 0.9617504 (1486)	total: 17m 41s	remaining: 0us

bestTest = 0.961750377
bestIteration = 1486

Shrink model to first 1487 iterations.

Depth 11
ROC-AUC: 0.961750
Best iteration: 1486
Time: 17.73 minutes

TESTING DEPTH = 12
0:	test: 0.9211990	best: 0.9211990

In [12]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import time

learning_rates = [0.03, 0.05, 0.08]

learning_rate_results = []

for lr in learning_rates:

    print("\n" + "="*60)
    print(f"TESTING LEARNING RATE = {lr}")
    print("="*60)

    start_time = time.time()

    model = CatBoostClassifier(
        iterations=1500,
        learning_rate=lr,
        depth=10,
        eval_metric="AUC",
        loss_function="Logloss",
        random_seed=42,
        verbose=500,
        allow_writing_files=False
    )

    model.fit(
        X_train_cb,
        y_train_cb,
        cat_features=categorical_features,
        eval_set=(X_val_cb, y_val_cb),
        use_best_model=True
    )

    val_proba = model.predict_proba(X_val_cb)[:, 1]

    auc = roc_auc_score(y_val_cb, val_proba)

    best_iteration = model.get_best_iteration()

    elapsed = (time.time() - start_time) / 60

    learning_rate_results.append({
        "learning_rate": lr,
        "roc_auc": auc,
        "best_iteration": best_iteration,
        "time_minutes": elapsed
    })

    print(f"\nLearning Rate: {lr}")
    print(f"ROC-AUC: {auc:.6f}")
    print(f"Best iteration: {best_iteration}")
    print(f"Time: {elapsed:.2f} minutes")


print("\n" + "="*60)
print("LEARNING RATE EXPERIMENT RESULTS")
print("="*60)

for result in learning_rate_results:
    print(
        f"LR {result['learning_rate']}: "
        f"AUC={result['roc_auc']:.6f}, "
        f"Best Iteration={result['best_iteration']}"
    )


TESTING LEARNING RATE = 0.03
0:	test: 0.9160770	best: 0.9160770 (0)	total: 682ms	remaining: 17m 2s
500:	test: 0.9517124	best: 0.9517124 (500)	total: 4m 57s	remaining: 9m 53s
1000:	test: 0.9579155	best: 0.9579155 (1000)	total: 9m 48s	remaining: 4m 53s
1499:	test: 0.9601021	best: 0.9601021 (1499)	total: 14m 41s	remaining: 0us

bestTest = 0.9601021245
bestIteration = 1499


Learning Rate: 0.03
ROC-AUC: 0.960102
Best iteration: 1499
Time: 14.73 minutes

TESTING LEARNING RATE = 0.05
0:	test: 0.9160770	best: 0.9160770 (0)	total: 627ms	remaining: 15m 40s
500:	test: 0.9565495	best: 0.9565495 (500)	total: 4m 55s	remaining: 9m 49s
1000:	test: 0.9605804	best: 0.9605804 (1000)	total: 9m 56s	remaining: 4m 57s
1499:	test: 0.9618250	best: 0.9618269 (1498)	total: 15m	remaining: 0us

bestTest = 0.9618269257
bestIteration = 1498

Shrink model to first 1499 iterations.

Learning Rate: 0.05
ROC-AUC: 0.961827
Best iteration: 1498
Time: 15.04 minutes

TESTING LEARNING RATE = 0.08
0:	test: 0.9160770	best: 0

In [14]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import time

iteration_values = [1500, 2000, 2500]

iteration_results = []

for iters in iteration_values:

    print("\n" + "="*60)
    print(f"TESTING ITERATIONS = {iters}")
    print("="*60)

    start_time = time.time()

    model = CatBoostClassifier(
        iterations=iters,
        learning_rate=0.08,
        depth=10,
        eval_metric="AUC",
        loss_function="Logloss",
        random_seed=42,
        verbose=500,
        allow_writing_files=False
    )

    model.fit(
        X_train_cb,
        y_train_cb,
        cat_features=categorical_features,
        eval_set=(X_val_cb, y_val_cb),
        use_best_model=True
    )

    val_proba = model.predict_proba(X_val_cb)[:, 1]

    auc = roc_auc_score(y_val_cb, val_proba)
    best_iteration = model.get_best_iteration()

    elapsed = (time.time() - start_time) / 60

    iteration_results.append({
        "iterations": iters,
        "roc_auc": auc,
        "best_iteration": best_iteration,
        "time_minutes": elapsed
    })

    print(f"\nIterations: {iters}")
    print(f"ROC-AUC: {auc:.6f}")
    print(f"Best iteration: {best_iteration}")
    print(f"Time: {elapsed:.2f} minutes")


print("\n" + "="*60)
print("ITERATION EXPERIMENT RESULTS")
print("="*60)

for result in iteration_results:
    print(
        f"Iterations {result['iterations']}: "
        f"AUC={result['roc_auc']:.6f}, "
        f"Best Iteration={result['best_iteration']}"
    )


TESTING ITERATIONS = 1500
0:	test: 0.9160770	best: 0.9160770 (0)	total: 682ms	remaining: 17m 2s
500:	test: 0.9596617	best: 0.9596617 (500)	total: 4m 57s	remaining: 9m 53s
1000:	test: 0.9617217	best: 0.9617217 (1000)	total: 9m 56s	remaining: 4m 57s
1499:	test: 0.9621947	best: 0.9621957 (1497)	total: 15m 1s	remaining: 0us

bestTest = 0.9621956975
bestIteration = 1497

Shrink model to first 1498 iterations.

Iterations: 1500
ROC-AUC: 0.962196
Best iteration: 1497
Time: 15.06 minutes

TESTING ITERATIONS = 2000
0:	test: 0.9160770	best: 0.9160770 (0)	total: 605ms	remaining: 20m 9s
500:	test: 0.9596617	best: 0.9596617 (500)	total: 4m 53s	remaining: 14m 39s
1000:	test: 0.9617217	best: 0.9617217 (1000)	total: 9m 53s	remaining: 9m 52s
1500:	test: 0.9621935	best: 0.9621957 (1497)	total: 14m 57s	remaining: 4m 58s
1999:	test: 0.9621719	best: 0.9622384 (1684)	total: 20m 4s	remaining: 0us

bestTest = 0.9622384432
bestIteration = 1684

Shrink model to first 1685 iterations.

Iterations: 2000
ROC-AUC:

In [15]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import time

# ============================================================
# FINAL PARAMETER EXPERIMENT: L2 LEAF REGULARIZATION
# ============================================================

l2_values = [1, 3, 5, 10]

l2_results = []

for l2 in l2_values:

    print("\n" + "=" * 60)
    print(f"TESTING L2_LEAF_REG = {l2}")
    print("=" * 60)

    start_time = time.time()

    model = CatBoostClassifier(
        iterations=1685,
        learning_rate=0.08,
        depth=10,
        l2_leaf_reg=l2,

        loss_function="Logloss",
        eval_metric="AUC",

        random_seed=42,
        verbose=500,
        allow_writing_files=False
    )

    model.fit(
        X_train_cb,
        y_train_cb,

        cat_features=categorical_features,

        eval_set=(X_val_cb, y_val_cb),
        use_best_model=True
    )

    # Validation predictions
    val_probs = model.predict_proba(X_val_cb)[:, 1]

    auc = roc_auc_score(y_val_cb, val_probs)

    best_iteration = model.get_best_iteration()

    elapsed = (time.time() - start_time) / 60

    print(f"\nL2 Leaf Reg: {l2}")
    print(f"ROC-AUC: {auc:.6f}")
    print(f"Best iteration: {best_iteration}")
    print(f"Time: {elapsed:.2f} minutes")

    l2_results.append({
        "l2_leaf_reg": l2,
        "roc_auc": auc,
        "best_iteration": best_iteration,
        "time_minutes": elapsed
    })


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 60)
print("L2 LEAF REGULARIZATION RESULTS")
print("=" * 60)

for result in l2_results:
    print(
        f"L2={result['l2_leaf_reg']}: "
        f"AUC={result['roc_auc']:.6f}, "
        f"Best Iteration={result['best_iteration']}"
    )

# Best configuration
best_l2_result = max(l2_results, key=lambda x: x["roc_auc"])

print("\n" + "=" * 60)
print("BEST L2 CONFIGURATION")
print("=" * 60)

print(f"Best L2 Leaf Reg: {best_l2_result['l2_leaf_reg']}")
print(f"Best Validation AUC: {best_l2_result['roc_auc']:.6f}")
print(f"Best Iteration: {best_l2_result['best_iteration']}")


TESTING L2_LEAF_REG = 1
0:	test: 0.9160770	best: 0.9160770 (0)	total: 604ms	remaining: 16m 56s
500:	test: 0.9594504	best: 0.9594504 (500)	total: 4m 54s	remaining: 11m 35s
1000:	test: 0.9617564	best: 0.9617590 (992)	total: 9m 55s	remaining: 6m 46s
1500:	test: 0.9621762	best: 0.9621825 (1384)	total: 14m 58s	remaining: 1m 50s
1684:	test: 0.9622355	best: 0.9622355 (1684)	total: 16m 52s	remaining: 0us

bestTest = 0.9622354721
bestIteration = 1684


L2 Leaf Reg: 1
ROC-AUC: 0.962235
Best iteration: 1684
Time: 16.91 minutes

TESTING L2_LEAF_REG = 3
0:	test: 0.9160770	best: 0.9160770 (0)	total: 609ms	remaining: 17m 5s
500:	test: 0.9596617	best: 0.9596617 (500)	total: 4m 54s	remaining: 11m 36s
1000:	test: 0.9617217	best: 0.9617217 (1000)	total: 9m 53s	remaining: 6m 45s
1500:	test: 0.9621935	best: 0.9621957 (1497)	total: 14m 58s	remaining: 1m 50s
1684:	test: 0.9622384	best: 0.9622384 (1684)	total: 16m 50s	remaining: 0us

bestTest = 0.9622384432
bestIteration = 1684


L2 Leaf Reg: 3
ROC-AUC: 0.96

In [7]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import time

# ============================================================
# FINAL 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

N_SPLITS = 5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)

fold_auc_scores = []
fold_times = []

print("=" * 60)
print("FINAL 5-FOLD CROSS-VALIDATION")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train_cb, y_train_cb), start=1
):

    print("\n" + "=" * 60)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 60)

    start_time = time.time()

    # Split the already-prepared training data
    X_fold_train = X_train_cb.iloc[train_idx].copy()
    X_fold_val   = X_train_cb.iloc[val_idx].copy()

    y_fold_train = y_train_cb.iloc[train_idx]
    y_fold_val   = y_train_cb.iloc[val_idx]

    # --------------------------------------------------------
    # Create model with FINAL selected parameters
    # --------------------------------------------------------

    model = CatBoostClassifier(
        iterations=1685,
        learning_rate=0.08,
        depth=10,
        l2_leaf_reg=10,

        loss_function="Logloss",
        eval_metric="AUC",

        random_seed=42,
        verbose=300,
        allow_writing_files=False
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model.fit(
        X_fold_train,
        y_fold_train,

        cat_features=categorical_features,

        eval_set=(X_fold_val, y_fold_val),

        use_best_model=True
    )

    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------

    val_prob = model.predict_proba(X_fold_val)[:, 1]

    fold_auc = roc_auc_score(
        y_fold_val,
        val_prob
    )

    elapsed = (time.time() - start_time) / 60

    fold_auc_scores.append(fold_auc)
    fold_times.append(elapsed)

    print(f"\nFold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} time: {elapsed:.2f} minutes")
    print(f"Best iteration: {model.get_best_iteration()}")


# ============================================================
# FINAL CV RESULTS
# ============================================================

mean_auc = np.mean(fold_auc_scores)
std_auc = np.std(fold_auc_scores, ddof=1)

print("\n" + "=" * 60)
print("FINAL CROSS-VALIDATION RESULTS")
print("=" * 60)

for i, score in enumerate(fold_auc_scores, start=1):
    print(f"Fold {i}: {score:.6f}")

print("-" * 60)
print(f"Mean ROC-AUC: {mean_auc:.6f}")
print(f"Std ROC-AUC:  {std_auc:.6f}")
print(f"Min ROC-AUC:  {min(fold_auc_scores):.6f}")
print(f"Max ROC-AUC:  {max(fold_auc_scores):.6f}")

print(f"\nAverage fold time: {np.mean(fold_times):.2f} minutes")
print(f"Total CV time: {np.sum(fold_times):.2f} minutes")

FINAL 5-FOLD CROSS-VALIDATION

FOLD 1/5
0:	test: 0.9181654	best: 0.9181654 (0)	total: 685ms	remaining: 19m 13s
300:	test: 0.9559315	best: 0.9559315 (300)	total: 2m 45s	remaining: 12m 41s
600:	test: 0.9597620	best: 0.9597620 (600)	total: 5m 28s	remaining: 9m 52s
900:	test: 0.9609142	best: 0.9609142 (899)	total: 8m 10s	remaining: 7m 6s
1200:	test: 0.9614430	best: 0.9614452 (1198)	total: 10m 55s	remaining: 4m 24s
1500:	test: 0.9615665	best: 0.9615690 (1458)	total: 13m 43s	remaining: 1m 40s
1684:	test: 0.9615622	best: 0.9615810 (1519)	total: 15m 26s	remaining: 0us

bestTest = 0.9615810368
bestIteration = 1519

Shrink model to first 1520 iterations.

Fold 1 ROC-AUC: 0.961581
Fold 1 time: 15.48 minutes
Best iteration: 1519

FOLD 2/5
0:	test: 0.9184257	best: 0.9184257 (0)	total: 580ms	remaining: 16m 17s
300:	test: 0.9555509	best: 0.9555509 (300)	total: 2m 42s	remaining: 12m 25s
600:	test: 0.9598356	best: 0.9598356 (600)	total: 5m 25s	remaining: 9m 46s
900:	test: 0.9613641	best: 0.9613641 (900

In [10]:
from catboost import CatBoostClassifier
import time

# ============================================================
# FINAL MODEL — TRAIN ON 100% OF TRAINING DATA
# ============================================================

FINAL_ITERATIONS = 1562

print("=" * 60)
print("TRAINING FINAL CATBOOST MODEL")
print("=" * 60)

final_model = CatBoostClassifier(
    iterations=FINAL_ITERATIONS,
    learning_rate=0.08,
    depth=10,
    l2_leaf_reg=10,

    loss_function="Logloss",
    eval_metric="AUC",

    random_seed=42,

    verbose=300,
    allow_writing_files=False
)

start_time = time.time()

final_model.fit(
    X_train_cb,
    y_train_cb,
    cat_features=categorical_features
)

elapsed = (time.time() - start_time) / 60

print("\n" + "=" * 60)
print("FINAL MODEL TRAINING COMPLETE")
print("=" * 60)

print(f"Iterations: {FINAL_ITERATIONS}")
print("Learning Rate: 0.08")
print("Depth: 10")
print("L2 Leaf Reg: 10")
print(f"Training time: {elapsed:.2f} minutes")

TRAINING FINAL CATBOOST MODEL
0:	total: 661ms	remaining: 17m 11s
300:	total: 3m 10s	remaining: 13m 16s
600:	total: 6m 21s	remaining: 10m 10s
900:	total: 9m 35s	remaining: 7m 2s
1200:	total: 12m 46s	remaining: 3m 50s
1500:	total: 15m 55s	remaining: 38.8s
1561:	total: 16m 34s	remaining: 0us

FINAL MODEL TRAINING COMPLETE
Iterations: 1562
Learning Rate: 0.08
Depth: 10
L2 Leaf Reg: 10
Training time: 16.59 minutes


In [12]:
# ============================================================
# PREDICTIONS ON UNSEEN TEST DATA
# ============================================================

print("=" * 60)
print("GENERATING TEST PREDICTIONS")
print("=" * 60)

# Probability of class 1
test_probabilities = final_model.predict_proba(X_test_cb)[:, 1]

# Class predictions using default 0.5 threshold
test_predictions = (test_probabilities >= 0.5).astype(int)

print("Test predictions generated.")
print(f"Number of test samples: {len(test_predictions)}")
print(f"Predicted class 0: {(test_predictions == 0).sum()}")
print(f"Predicted class 1: {(test_predictions == 1).sum()}")

GENERATING TEST PREDICTIONS
Test predictions generated.
Number of test samples: 296302
Predicted class 0: 84008
Predicted class 1: 212294


In [13]:
# ============================================================
# CREATE SUBMISSION FILE
# ============================================================

submission = pd.DataFrame({
    "id": test["id"],
    "target": test_predictions
})

submission.to_csv("submission.csv", index=False)

print("=" * 60)
print("SUBMISSION CREATED")
print("=" * 60)

print(submission.head())
print(f"\nSubmission shape: {submission.shape}")
print("\nPrediction distribution:")
print(submission["target"].value_counts())

print("\nSaved as: submission.csv")

SUBMISSION CREATED
       id  target
0  691369       1
1  691370       1
2  691371       1
3  691372       1
4  691373       1

Submission shape: (296302, 2)

Prediction distribution:
target
1    212294
0     84008
Name: count, dtype: int64

Saved as: submission.csv
